# TP Python 3 : Raster & Apprentissage automatique (MLP)

**Géoinformatique II — Université de Lausanne**

---

Ce TP introduit deux nouvelles dimensions de la géoinformatique en Python :

| Partie | Bibliothèque | Ce que tu apprendras |
|--------|-------------|----------------------|
| **1. Données raster** | `rasterio` | Lire, explorer, visualiser et manipuler des fichiers raster |
| **2. Réseaux de neurones** | `scikit-learn` | Entraîner un perceptron multicouche (MLP) pour classifier des pixels |

> 🔗 **Lien avec les TPs précédents** : jusqu'ici tu as travaillé avec des
> données **vectorielles** (points, lignes, polygones). Les rasters sont l'autre
> grande famille de données géospatiales, complémentaire aux vecteurs.

In [ ]:
# Importation de toutes les bibliothèques nécessaires
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

import rasterio
from rasterio.transform import from_bounds
from rasterio.crs import CRS
from rasterio.plot import show, show_hist

import geopandas as gpd
from shapely.geometry import box

from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import warnings
warnings.filterwarnings('ignore')

print("Bibliothèques chargées :")
print(f"  rasterio   : {rasterio.__version__}")
print(f"  scikit-learn importé avec succès")

---
## Partie 1 — Rasterio : données raster

### 1.1 Qu'est-ce qu'un raster ?

Un **raster** est une représentation du monde sous forme de **grille régulière de pixels**.
Chaque pixel a :
- une **position** (déterminée par le CRS, l'emprise et la résolution)
- une ou plusieurs **valeurs** (une par bande spectrale)

```
┌───┬───┬───┬───┐
│255│200│180│ 90│  ← ligne 0
├───┼───┼───┼───┤
│210│195│ 85│ 40│  ← ligne 1
├───┼───┼───┼───┤
│ 20│ 30│ 50│ 60│  ← ligne 2
└───┴───┴───┴───┘
```

| Concept raster | Équivalent numérique |
|---|---|
| Image à 1 bande | Tableau NumPy 2D de forme `(lignes, colonnes)` |
| Image à N bandes | Tableau NumPy 3D de forme `(N, lignes, colonnes)` |
| Résolution | Taille d'un pixel en unités du CRS (ex: 10 m × 10 m) |
| Emprise (extent) | `(xmin, ymin, xmax, ymax)` en coordonnées géographiques |
| Valeur NoData | Pixel hors zone d'intérêt (ex: `np.nan` ou `-9999`) |

> 🔗 **Exemples de rasters en géographie** : modèles numériques de terrain (MNT),
> images satellites (Sentinel-2, Landsat), cartes de précipitations, cartes d'occupation du sol.

### 1.2 Créer et écrire un raster synthétique

Avant de lire des fichiers réels, on va **créer un raster depuis zéro** avec NumPy et rasterio.
Cela permet de comprendre la structure interne d'un fichier GeoTIFF.

In [ ]:
import os
os.makedirs("rasters", exist_ok=True)

# -------------------------------------------------------
# Paramètres du raster
# -------------------------------------------------------
np.random.seed(42)
ROWS, COLS = 200, 200          # résolution en pixels
NODATA     = -9999.0

# Emprise : coin sud-ouest du canton de Vaud (coordonnées MN03 approximatives)
# xmin, ymin, xmax, ymax en mètres (EPSG:21781)
XMIN, YMIN, XMAX, YMAX = 500_000, 130_000, 600_000, 190_000

# La transformation geospatiale lie les indices (ligne, col) aux coordonnées réelles
transform = from_bounds(XMIN, YMIN, XMAX, YMAX, width=COLS, height=ROWS)

# -------------------------------------------------------
# Génération d'un MNT (modèle numérique de terrain) synthétique
# -------------------------------------------------------
# On simule un relief avec des gradients et du bruit
x = np.linspace(0, 2 * np.pi, COLS)
y = np.linspace(0, 2 * np.pi, ROWS)
XX, YY = np.meshgrid(x, y)

# Relief principal (crête est-ouest)
dem = (
    800  +
    400  * np.sin(0.5 * XX) * np.cos(0.3 * YY) +
    200  * np.sin(1.2 * YY) +
    100  * np.random.randn(ROWS, COLS)   # bruit terrain
)
dem = dem.clip(0, 3000).astype(np.float32)

# -------------------------------------------------------
# Écriture du fichier GeoTIFF
# -------------------------------------------------------
MNT_PATH = "rasters/mnt_vaud_synth.tif"

with rasterio.open(
    MNT_PATH,
    mode='w',
    driver='GTiff',
    height=ROWS,
    width=COLS,
    count=1,                    # 1 bande
    dtype='float32',
    crs=CRS.from_epsg(21781),   # MN03
    transform=transform,
    nodata=NODATA,
) as dst:
    dst.write(dem, 1)           # bande 1 = dem

print(f"Raster MNT créé : {MNT_PATH}")
print(f"  Dimensions      : {ROWS} × {COLS} pixels")
print(f"  Résolution pixel: {(XMAX-XMIN)/COLS:.0f} m × {(YMAX-YMIN)/ROWS:.0f} m")
print(f"  Altitude min/max: {dem.min():.0f} m / {dem.max():.0f} m")

### 1.3 Lire et explorer les métadonnées d'un raster

`rasterio.open()` retourne un objet **dataset** qui expose toutes les métadonnées
du fichier — exactement comme les *Propriétés de couche* dans QGIS, onglet *Information*.

In [ ]:
# Ouverture du fichier en lecture
with rasterio.open(MNT_PATH) as src:

    print("=== Métadonnées du raster ===")
    print(f"Pilote (format)    : {src.driver}")
    print(f"Dimensions         : {src.height} lignes × {src.width} colonnes")
    print(f"Nombre de bandes   : {src.count}")
    print(f"Type de données    : {src.dtypes[0]}")
    print(f"CRS                : {src.crs}")
    print(f"Code EPSG          : {src.crs.to_epsg()}")
    print(f"Valeur NoData      : {src.nodata}")
    print()

    # L'emprise (bounds) donne les coordonnées géographiques des bords du raster
    bounds = src.bounds
    print(f"Emprise (bounds)   :")
    print(f"  West  (xmin) = {bounds.left:.0f} m")
    print(f"  East  (xmax) = {bounds.right:.0f} m")
    print(f"  South (ymin) = {bounds.bottom:.0f} m")
    print(f"  North (ymax) = {bounds.top:.0f} m")
    print()

    # La transformation géoréférence les pixels
    print(f"Transform (affine) :\n{src.transform}")
    print()
    # Résolution d'un pixel
    res_x, res_y = src.res
    print(f"Résolution         : {res_x:.1f} m × {res_y:.1f} m par pixel")

    # Lecture de la bande 1 comme tableau NumPy
    data = src.read(1)

print(f"\nTableau NumPy : shape={data.shape}, dtype={data.dtype}")
print(f"Valeurs       : min={data.min():.1f}, max={data.max():.1f}, mean={data.mean():.1f}")

### 1.4 Visualiser un raster

`rasterio.plot.show()` gère automatiquement les coordonnées géographiques sur les axes.

In [ ]:
with rasterio.open(MNT_PATH) as src:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # --- Visualisation avec rasterio.plot.show() ---
    # Gère automatiquement le géoréférencement sur les axes
    show(src, ax=axes[0], cmap='terrain', title='MNT synthétique (rasterio.plot.show)')
    axes[0].set_xlabel('Est (m)')
    axes[0].set_ylabel('Nord (m)')

    # --- Visualisation avec matplotlib imshow + extent ---
    # Permet plus de contrôle sur la colorbar et le style
    data = src.read(1)
    bounds = src.bounds
    extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]

    im = axes[1].imshow(data, cmap='terrain', extent=extent, origin='upper')
    plt.colorbar(im, ax=axes[1], label='Altitude (m)', shrink=0.8)
    axes[1].set_title('MNT synthétique (imshow + extent)')
    axes[1].set_xlabel('Est (m)')
    axes[1].set_ylabel('Nord (m)')

plt.tight_layout()
plt.show()

### 1.5 Raster multi-bandes : image multispectrale simulée

En télédétection, une image satellite contient plusieurs **bandes spectrales**
(rouge, vert, bleu, proche infrarouge…). Rasterio les stocke dans un tableau 3D :

```
bandes × lignes × colonnes  →  shape (4, 200, 200) pour une image 4 bandes
```

On va simuler une image multispectrale à 4 bandes calée sur le même MNT :
- **Bande 1** — Bleu (B) 
- **Bande 2** — Vert (G)
- **Bande 3** — Rouge (R)
- **Bande 4** — Proche infrarouge (NIR)

In [ ]:
np.random.seed(0)

# On crée un masque d'occupation du sol (ground truth) basé sur l'altitude
# pour rendre les bandes spectrales réalistes
#   0 = eau        (zones basses, < 400 m)
#   1 = prairie    (400–900 m)
#   2 = forêt      (900–1800 m)
#   3 = roche/neige (> 1800 m)

CLASSES   = {0: 'Eau',  1: 'Prairie', 2: 'Forêt', 3: 'Roche/Neige'}
CLASS_COLORS = ['#4fc3f7', '#a5d6a7', '#2e7d32', '#bdbdbd']

gt = np.zeros((ROWS, COLS), dtype=np.uint8)
gt[dem < 400]                     = 0   # eau
gt[(dem >= 400) & (dem < 900)]    = 1   # prairie
gt[(dem >= 900) & (dem < 1800)]   = 2   # forêt
gt[dem >= 1800]                   = 3   # roche

# Signatures spectrales moyennes (valeurs 0–1 normalisées) par classe
#          B      G      R     NIR
SIG = np.array([
    [0.06, 0.12, 0.08, 0.04],   # eau      : forte absorption NIR
    [0.08, 0.20, 0.12, 0.55],   # prairie  : fort NIR (végétation)
    [0.05, 0.15, 0.07, 0.70],   # forêt    : très fort NIR
    [0.35, 0.35, 0.38, 0.30],   # roche    : réflectance uniforme
])

# Construction des 4 bandes avec bruit radiométrique
bandes = np.zeros((4, ROWS, COLS), dtype=np.float32)
for c in range(4):
    for cls in range(4):
        bandes[c][gt == cls] = SIG[cls, c]
    bandes[c] += 0.02 * np.random.randn(ROWS, COLS).astype(np.float32)  # bruit capteur
    bandes[c] = np.clip(bandes[c], 0, 1)

print(f"Image multispectrale : shape={bandes.shape}  dtype={bandes.dtype}")
for i, nom in enumerate(['Bleu', 'Vert', 'Rouge', 'NIR']):
    print(f"  Bande {i+1} ({nom:5s}) : min={bandes[i].min():.3f}, max={bandes[i].max():.3f}")

In [ ]:
# Écriture du GeoTIFF multi-bandes
MS_PATH = "rasters/multispectral_synth.tif"

with rasterio.open(
    MS_PATH, 'w',
    driver='GTiff',
    height=ROWS, width=COLS,
    count=4,                     # 4 bandes
    dtype='float32',
    crs=CRS.from_epsg(21781),
    transform=transform,
) as dst:
    dst.write(bandes)            # écrit les 4 bandes d'un coup

print(f"Image multispectrale sauvegardée : {MS_PATH}")

# -------------------------------------------------------
# Visualisation : composition colorée RGB + occupation du sol
# -------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Composition RGB (bandes R=3, G=2, B=1 → indices 2, 1, 0)
rgb = np.dstack([bandes[2], bandes[1], bandes[0]])
rgb_norm = (rgb - rgb.min()) / (rgb.max() - rgb.min())  # normalisation 0–1

extent = [XMIN, XMAX, YMIN, YMAX]
axes[0].imshow(rgb_norm, extent=extent, origin='upper')
axes[0].set_title('Composition colorée RGB (R=bande3, G=bande2, B=bande1)', fontsize=10)
axes[0].set_xlabel('Est (m)')
axes[0].set_ylabel('Nord (m)')

# Carte d'occupation du sol (vérité terrain)
cmap_gt  = mcolors.ListedColormap(CLASS_COLORS)
norm_gt  = mcolors.BoundaryNorm([0, 1, 2, 3, 4], cmap_gt.N)
im = axes[1].imshow(gt, cmap=cmap_gt, norm=norm_gt, extent=extent, origin='upper')
patches = [Patch(color=CLASS_COLORS[i], label=CLASSES[i]) for i in range(4)]
axes[1].legend(handles=patches, loc='lower right', fontsize=9, title='Occupation du sol')
axes[1].set_title('Occupation du sol (vérité terrain)', fontsize=10)
axes[1].set_xlabel('Est (m)')
axes[1].set_ylabel('Nord (m)')

plt.tight_layout()
plt.show()

### 1.6 Calcul d'indices spectraux — NDVI

Le **NDVI** (Normalized Difference Vegetation Index) est l'un des indices
les plus utilisés en télédétection pour quantifier la densité de végétation :

$$\text{NDVI} = \frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red}}$$

- NDVI ≈ **−1 à 0** : eau, roche, neige  
- NDVI ≈ **0 à 0.3** : sol nu, prairie clairsemée  
- NDVI ≈ **0.3 à 1** : végétation dense (forêt, prairie)

> 🔗 **En QGIS** : outil *Calculatrice raster* avec la formule `(NIR - Red) / (NIR + Red)`.

In [ ]:
with rasterio.open(MS_PATH) as src:
    rouge = src.read(3).astype(np.float32)   # bande 3 = Rouge
    nir   = src.read(4).astype(np.float32)   # bande 4 = NIR

# Calcul NDVI — protection division par zéro
with np.errstate(divide='ignore', invalid='ignore'):
    ndvi = np.where(
        (nir + rouge) == 0,
        np.nan,
        (nir - rouge) / (nir + rouge)
    )

print(f"NDVI : min={np.nanmin(ndvi):.3f}, max={np.nanmax(ndvi):.3f}, mean={np.nanmean(ndvi):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Carte NDVI
im = axes[0].imshow(ndvi, cmap='RdYlGn', vmin=-0.2, vmax=0.8,
                    extent=extent, origin='upper')
plt.colorbar(im, ax=axes[0], label='NDVI', shrink=0.8)
axes[0].set_title('NDVI — indice de végétation', fontsize=11)
axes[0].set_xlabel('Est (m)')
axes[0].set_ylabel('Nord (m)')

# Histogramme des valeurs NDVI
axes[1].hist(ndvi[~np.isnan(ndvi)].ravel(), bins=60, color='#388e3c', alpha=0.8, edgecolor='white')
axes[1].axvline(0,   color='#1565c0', linestyle='--', label='NDVI = 0')
axes[1].axvline(0.3, color='#e65100', linestyle='--', label='NDVI = 0.3')
axes[1].set_title('Distribution des valeurs NDVI', fontsize=11)
axes[1].set_xlabel('NDVI')
axes[1].set_ylabel('Nombre de pixels')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Partie 2 — Perceptron multicouche (MLP) avec scikit-learn

### 2.1 Architecture d'un réseau de neurones

Un **perceptron multicouche (MLP)** est un réseau de neurones artificiels composé de
**couches** (`layers`) de **neurones** (`units`) connectées les unes aux autres.

```
Entrée (features)       Couche cachée 1     Couche cachée 2     Sortie (classe)
  B, G, R, NIR  →  [●●●●●●●●●●]  →  [●●●●●●]  →  [classe prédite]
   (4 valeurs)          (64 unités)    (32 unités)   (4 probabilités)
```

**Fonctionnement** :
1. Chaque neurone calcule la somme pondérée de ses entrées + un biais
2. Il applique une **fonction d'activation** non-linéaire (ex: ReLU, tanh)
3. Le résultat se propage vers la couche suivante

**Entraînement** : on ajuste les poids par **rétropropagation du gradient** pour
minimiser une fonction de perte (cross-entropy pour la classification).

| Paramètre scikit-learn | Description |
|---|---|
| `hidden_layer_sizes=(64, 32)` | 2 couches cachées : 64 puis 32 neurones |
| `activation='relu'` | Fonction d'activation ReLU |
| `max_iter=200` | Nombre maximum d'époques |
| `learning_rate_init=0.001` | Taux d'apprentissage initial |
| `random_state=42` | Reproductibilité |

### 2.2 Préparation des données

Le MLP attend un tableau 2D `X` de shape `(n_pixels, n_features)`.
On doit **aplatir** (reshape) les tableaux raster pour transformer les images
en tableau de pixels :

In [ ]:
# -------------------------------------------------------
# De l'image raster au tableau scikit-learn
# -------------------------------------------------------

# bandes : shape (4, 200, 200)   →  tableau 3D (bandes, lignes, colonnes)
# On veut  : shape (40000, 4)    →  tableau 2D (pixels, features)

n_pixels = ROWS * COLS
n_bandes = bandes.shape[0]

# Transposition puis reshape :  (4, 200, 200)  →  (200*200, 4)
X = bandes.reshape(n_bandes, n_pixels).T          # shape : (40000, 4)
y = gt.ravel()                                     # shape : (40000,)

# Ajout du NDVI comme 5e feature
ndvi_flat = ndvi.ravel()
X = np.hstack([X, ndvi_flat.reshape(-1, 1)])       # shape : (40000, 5)

print(f"X (features) : shape={X.shape}  — {n_pixels} pixels × {X.shape[1]} features")
print(f"y (labels)   : shape={y.shape}  — valeurs uniques : {np.unique(y)}")
print()
print(f"Distribution des classes :")
for cls, nom in CLASSES.items():
    n = (y == cls).sum()
    print(f"  Classe {cls} ({nom:12s}) : {n:6d} pixels  ({100*n/n_pixels:.1f} %)")

In [ ]:
# -------------------------------------------------------
# Séparation entraînement / test
# -------------------------------------------------------
# On n'utilise qu'un échantillon de pixels pour accélérer l'entraînement
# (en pratique on travaillerait avec tous les pixels)
np.random.seed(42)
idx = np.random.choice(n_pixels, size=10_000, replace=False)   # 10 000 pixels

X_sample = X[idx]
y_sample = y[idx]

X_train, X_test, y_train, y_test = train_test_split(
    X_sample, y_sample, test_size=0.2, random_state=42, stratify=y_sample
)

print(f"Échantillon total : {len(X_sample)} pixels")
print(f"  Entraînement    : {len(X_train)} pixels  ({100*len(X_train)/len(X_sample):.0f} %)")
print(f"  Test            : {len(X_test)}  pixels  ({100*len(X_test)/len(X_sample):.0f} %)")
print()

# -------------------------------------------------------
# Normalisation des features (OBLIGATOIRE pour les réseaux de neurones)
# -------------------------------------------------------
# StandardScaler centre et réduit chaque feature : μ=0, σ=1
# Important : on ajuste le scaler UNIQUEMENT sur l'entraînement !
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)    # ajustement + transformation
X_test_sc  = scaler.transform(X_test)         # transformation seulement

print("Features après normalisation (moyennes) :")
print(np.round(X_train_sc.mean(axis=0), 4))   # doivent être ≈ 0
print("Features après normalisation (écarts-types) :")
print(np.round(X_train_sc.std(axis=0), 4))    # doivent être ≈ 1

### 2.3 Entraînement du MLP

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),   # 2 couches cachées
    activation='relu',             # ReLU : max(0, x) — standard pour les MLP
    solver='adam',                 # optimiseur Adam (recommandé)
    learning_rate_init=0.001,
    max_iter=300,
    early_stopping=True,           # arrêt si la validation ne s'améliore plus
    validation_fraction=0.1,       # 10 % de l'entraînement pour la validation
    n_iter_no_change=15,
    random_state=42,
    verbose=False,
)

mlp.fit(X_train_sc, y_train)

# Scores
train_acc = mlp.score(X_train_sc, y_train)
test_acc  = mlp.score(X_test_sc,  y_test)

print(f"Entraînement terminé en {mlp.n_iter_} itérations")
print(f"  Précision entraînement : {train_acc:.4f}  ({train_acc*100:.2f} %)")
print(f"  Précision test         : {test_acc:.4f}  ({test_acc*100:.2f} %)")

In [ ]:
# --- Courbe de perte (loss curve) ---
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(mlp.loss_curve_, label='Perte entraînement', color='#1565c0', linewidth=2)
if hasattr(mlp, 'validation_scores_') and mlp.validation_scores_ is not None:
    # Conversion score → perte approx. (1 - accuracy)
    ax.plot([1 - s for s in mlp.validation_scores_],
            label='Perte validation (approx.)', color='#e65100',
            linewidth=2, linestyle='--')
ax.set_title('Courbe de perte — MLP (classification occupation du sol)', fontsize=12)
ax.set_xlabel('Époque')
ax.set_ylabel('Perte (cross-entropy)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 2.4 Évaluation — rapport de classification et matrice de confusion

La **matrice de confusion** montre combien de pixels de chaque classe vraie
ont été prédits dans chaque classe prédite — ce qui permet de voir quelles
classes sont confondues entre elles.

In [ ]:
y_pred = mlp.predict(X_test_sc)
noms_classes = [CLASSES[i] for i in sorted(CLASSES)]

# Rapport textuel
print("=== Rapport de classification ===")
print(classification_report(y_test, y_pred, target_names=noms_classes))

# Matrice de confusion
fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=noms_classes)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Matrice de confusion — MLP (données test)', fontsize=12)
plt.tight_layout()
plt.show()

### 2.5 Carte de classification

On applique maintenant le modèle entraîné à **tous les pixels** de l'image
pour produire une carte d'occupation du sol classifiée.

In [ ]:
# Normaliser TOUS les pixels avec le même scaler entraîné
X_full_sc = scaler.transform(X)                          # shape : (40000, 5)

# Prédiction sur l'image complète
y_pred_full = mlp.predict(X_full_sc)                     # shape : (40000,)
carte_classif = y_pred_full.reshape(ROWS, COLS)          # shape : (200, 200)

# -------------------------------------------------------
# Comparaison : vérité terrain vs. carte classifiée
# -------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

cmap_gt = mcolors.ListedColormap(CLASS_COLORS)
norm_gt = mcolors.BoundaryNorm([0, 1, 2, 3, 4], cmap_gt.N)

for ax, data, title in [
    (axes[0], gt,           'Vérité terrain'),
    (axes[1], carte_classif, 'Carte classifiée par MLP'),
]:
    ax.imshow(data, cmap=cmap_gt, norm=norm_gt, extent=extent, origin='upper')
    patches = [Patch(color=CLASS_COLORS[i], label=CLASSES[i]) for i in range(4)]
    ax.legend(handles=patches, loc='lower right', fontsize=9, title='Occupation du sol')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Est (m)')
    ax.set_ylabel('Nord (m)')

plt.suptitle(f"Classification par MLP — précision globale = {test_acc*100:.1f} %",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.6 Régression — prédire l'altitude depuis les valeurs spectrales

Un **MLPRegressor** prédit une valeur continue (ici : l'altitude du MNT)
à partir des bandes spectrales et du NDVI.  
C'est un exemple de **régression** (par opposition à la classification).

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

# Cible : altitude (MNT aplati)
y_alt = dem.ravel()                          # shape : (40000,)

# Même échantillon que précédemment pour la cohérence
X_alt_train, X_alt_test, y_alt_train, y_alt_test = train_test_split(
    X[idx], y_alt[idx], test_size=0.2, random_state=42
)

scaler_reg = StandardScaler()
X_alt_train_sc = scaler_reg.fit_transform(X_alt_train)
X_alt_test_sc  = scaler_reg.transform(X_alt_test)

regr = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    learning_rate_init=0.001,
    max_iter=300,
    early_stopping=True,
    n_iter_no_change=15,
    random_state=42,
)
regr.fit(X_alt_train_sc, y_alt_train)

y_alt_pred = regr.predict(X_alt_test_sc)
mae = mean_absolute_error(y_alt_test, y_alt_pred)
r2  = r2_score(y_alt_test, y_alt_pred)

print(f"Régression MLP — prédiction d'altitude")
print(f"  MAE (erreur absolue moyenne) : {mae:.1f} m")
print(f"  R²                           : {r2:.4f}")

# ---- Nuage de points : valeurs réelles vs prédites ----
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_alt_test, y_alt_pred, s=5, alpha=0.4, color='#1565c0')
lim = [0, 3100]
ax.plot(lim, lim, color='red', linewidth=1.5, label='Prédiction parfaite')
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('Altitude réelle (m)')
ax.set_ylabel('Altitude prédite (m)')
ax.set_title(f'MLP Régression  —  MAE = {mae:.0f} m,  R² = {r2:.3f}', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Récapitulatif

### Partie 1 — Rasterio

| Opération | Code clé |
|---|---|
| Lire un raster | `rasterio.open(path)` → `.read(band)` |
| Métadonnées | `.meta`, `.crs`, `.transform`, `.bounds` |
| Écrire un raster | `rasterio.open(path, 'w', **meta)` → `.write(array, band)` |
| Visualiser | `rasterio.plot.show(src)` ou `plt.imshow(array)` |
| Indice spectral | opération pixel-à-pixel NumPy sur les bandes |

### Partie 2 — MLP avec scikit-learn

| Étape | Code clé |
|---|---|
| Préparation | `X = bandes.reshape(n_bandes, -1).T` |
| Split | `train_test_split(X, y, stratify=y)` |
| Normalisation | `StandardScaler().fit_transform(X_train)` |
| Classification | `MLPClassifier(...).fit(X_train_sc, y_train)` |
| Évaluation | `classification_report`, `confusion_matrix` |
| Carte classifiée | `clf.predict(X_full_sc).reshape(ROWS, COLS)` |
| Régression | `MLPRegressor(...).fit(X_train_sc, y_train)` |

> **À retenir** : la normalisation avec `StandardScaler` est
> **indispensable** pour les réseaux de neurones.
> Toujours ajuster (`fit`) sur l'entraînement uniquement,
> puis `transform` sur le test.